# Deep Learning 014 — Loss Functions

Companion notebook to the lesson. A loss function is the single number training is
trying to reduce; every gradient in the network is a derivative *of it*. Choosing it is
not a matter of taste — the task fixes it.

We build each one from its formula, plot its shape, and check the property that makes it
the right or wrong choice:

| Loss | Task | The property that decides it |
|---|---|---|
| MSE | regression | smooth everywhere, squares outliers |
| MAE | regression | outlier-robust, kinked at 0 |
| Huber | regression | MSE near zero, MAE far out |
| Binary cross-entropy | 2 classes | punishes confident mistakes without limit |
| Categorical cross-entropy | k classes | only the true class's probability matters |

`numpy` only; the plot cell needs `matplotlib` and can be skipped.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

## Loss vs cost — the confusion worth clearing first

- **Loss** is computed on **one** training example.
- **Cost** is the **average** of the loss over a batch or the whole dataset.

Everything below defines the per-example loss; the mean over an array is the cost. Most
libraries call both of them "loss", which is where the confusion comes from.

## Part A — Regression losses

$$\text{MSE} = (y - \hat y)^2 \qquad \text{MAE} = |y - \hat y| \qquad
\text{Huber}_\delta = \begin{cases} \tfrac12 e^2 & |e| \le \delta \\ \delta(|e| - \tfrac12\delta) & \text{otherwise}\end{cases}$$

In [ ]:
def mse(y, p):  return (y - p) ** 2
def mae(y, p):  return np.abs(y - p)
def huber(y, p, delta=1.0):
    e = np.abs(y - p)
    return np.where(e <= delta, 0.5 * e ** 2, delta * (e - 0.5 * delta))

err = np.array([0.0, 0.5, 1.0, 2.0, 5.0, 10.0])
print(f"{'error':>8}{'MSE':>10}{'MAE':>8}{'Huber':>9}")
for e in err:
    print(f"{e:>8.1f}{mse(0, e):>10.2f}{mae(0, e):>8.2f}{huber(0, e):>9.2f}")

Read the last row. An error of 10 costs MSE **100**, MAE **10**, Huber **9.5**.

That is the whole trade-off: MSE takes one bad outlier and makes it dominate the entire
dataset's gradient. Below is that claim as an experiment rather than an assertion — fit
the best constant predictor under each loss, on clean data and on the same data with one
point corrupted.

In [ ]:
clean = rng.normal(50, 5, size=200)
dirty = clean.copy()
dirty[0] = 500.0                                   # one typo in the data

grid = np.linspace(30, 90, 6001)
def best_constant(data, loss):
    costs = np.array([loss(data, c).mean() for c in grid])
    return grid[costs.argmin()]

print(f"{'':>10}{'clean':>10}{'with 1 outlier':>17}{'shift':>9}")
for name, loss in (("MSE", mse), ("MAE", mae), ("Huber", huber)):
    a, b = best_constant(clean, loss), best_constant(dirty, loss)
    print(f"{name:>10}{a:>10.2f}{b:>17.2f}{b - a:>9.2f}")
print(f"\ntrue mean of the clean data: {clean.mean():.2f}, median: {np.median(clean):.2f}")

**MSE moved 2.24 units because of one point out of two hundred. MAE did not move at
all.** Huber sits in between, which is what it is for.

The reason MAE is not simply the better choice: it has a **kink at zero**, so its
derivative jumps from −1 to +1 with nothing in between and never gets smaller as you
approach the answer. MSE's gradient shrinks as the error shrinks, which is a gentler
landing.

In [ ]:
# derivative of each loss w.r.t. the prediction, near zero error
h = 1e-6
for name, loss in (("MSE", mse), ("MAE", mae), ("Huber", huber)):
    grads = [(loss(0.0, e + h) - loss(0.0, e - h)) / (2 * h) for e in (-0.5, -0.01, 0.01, 0.5)]
    print(f"{name:>7}  d/dp at e = -0.5, -0.01, +0.01, +0.5: "
          + "  ".join(f"{g:+.3f}" for g in grads))

MSE's gradient goes to zero as the error does. MAE's is ±1 right up to the discontinuity —
it is not differentiable at 0, and that is the price of the robustness.

## Part B — Cross-entropy, and why it is not "just another distance"

For binary classification with a sigmoid output,

$$L = -\big[y\log \hat y + (1-y)\log(1 - \hat y)\big]$$

Only one term survives for any given example. If `y = 1` the loss is `-log(p)`; if `y = 0`
it is `-log(1-p)`.

In [ ]:
def bce(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

print("truth is 1:")
print(f"{'predicted p':>13}{'BCE':>9}{'MSE':>9}")
for p in (0.9, 0.7, 0.5, 0.3, 0.1, 0.01, 0.001):
    print(f"{p:>13.3f}{bce(1, p):>9.3f}{mse(1, p):>9.3f}")

Compare the two columns as the prediction gets worse.

- Predicting 0.9 when the truth is 1 costs BCE 0.105. Predicting 0.1 costs 2.303 —
  **about 20× more.** Under MSE the same pair costs 0.01 and 0.81, only 81×… but bounded.
- Predicting 0.001 costs BCE **6.9** and MSE **0.998**. MSE can never charge more than 1
  no matter how wrong and how confident the model is. **Cross-entropy is unbounded**, so a
  confidently wrong prediction produces a large gradient and gets corrected.

That is the argument for cross-entropy over MSE on classification, and it is a statement
about gradients, not about elegance.

In [ ]:
# Optional plot. Skip if matplotlib is unavailable.
import matplotlib.pyplot as plt

p = np.linspace(1e-3, 1 - 1e-3, 500)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(p, bce(1, p), label="BCE, truth = 1")
ax[0].plot(p, mse(1, p), "--", label="MSE, truth = 1")
ax[0].set(xlabel="predicted probability", ylabel="loss", title="confident and wrong")
ax[0].legend(); ax[0].grid(alpha=.3)

e = np.linspace(-4, 4, 500)
for name, loss in (("MSE", mse), ("MAE", mae), ("Huber", huber)):
    ax[1].plot(e, loss(0.0, e), label=name)
ax[1].set(xlabel="error", ylabel="loss", title="regression losses")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Part C — Categorical cross-entropy

With `k` classes the output is a softmax vector and the loss is

$$L = -\sum_{c} y_c \log \hat y_c$$

But `y` is one-hot, so every term except the true class is multiplied by zero. **Only the
probability the model assigned to the correct class matters.**

In [ ]:
def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()

def cce(y_onehot, p, eps=1e-12):
    return -(y_onehot * np.log(np.clip(p, eps, 1))).sum()

logits = np.array([2.0, 1.0, 0.1, -1.0])
p = softmax(logits)
print("probabilities:", np.round(p, 4), " sum =", p.sum())

for true_class in range(4):
    y = np.zeros(4); y[true_class] = 1
    print(f"  true class {true_class}: loss {cce(y, p):.4f}   "
          f"= -log({p[true_class]:.4f}) = {-np.log(p[true_class]):.4f}")

The loss is literally `-log(p_true)`. This is why the *sparse* version exists: if the
answer is always "the true class's probability", there is no reason to build a one-hot
vector at all — pass the class **index** and let the library index into `p`. Same loss,
less memory. `sparse_categorical_crossentropy` in Keras.

In [ ]:
def sparse_cce(idx, p, eps=1e-12):
    return -np.log(np.clip(p[idx], eps, 1))

for true_class in range(4):
    y = np.zeros(4); y[true_class] = 1
    assert np.isclose(cce(y, p), sparse_cce(true_class, p))
print("one-hot and sparse forms agree exactly for every class")

## Choosing, in one table

| Task | Output layer | Loss |
|---|---|---|
| regression | 1 node, linear | MSE (or MAE / Huber if outliers matter) |
| binary classification | 1 node, sigmoid | binary cross-entropy |
| multi-class, one-hot labels | k nodes, softmax | categorical cross-entropy |
| multi-class, integer labels | k nodes, softmax | sparse categorical cross-entropy |

The output activation and the loss are chosen **together**, and the task chooses both.

## Try it yourself

1. In Part A, vary the outlier from 100 to 5000. Plot the MSE-optimal constant against the
   outlier's size. Does it ever stop moving? Does MAE's?
2. Set `delta` in Huber to 0.1 and to 100. Which of MSE/MAE does each one become?
3. Show numerically that the constant minimising MSE is the **mean** and the constant
   minimising MAE is the **median**. (Part A almost says it; make it exact.)
4. Compute the gradient of BCE with respect to the *logit* rather than the probability for
   a sigmoid output. It simplifies to `p - y`. Verify that numerically — it is the reason
   sigmoid and BCE are always paired.